# OEC Phase-1 Login-Node Control Plane

This notebook is intentionally **login-node safe**.

Allowed here:
- lightweight quota metadata inspection with `quota_check`
- Slurm queue inspection
- creation of quota snapshots from already-parsed numbers
- preparation/submission of compute-node jobs

Not allowed here:
- CUDA/GPU inference
- model downloads
- dataset preprocessing
- package builds
- repository cloning
- training or other heavy computation

Run `phase1_test_compute_node.ipynb` inside an allocated compute job for GPU/model tests.


In [ ]:
import os
import subprocess
import core_config

if os.environ.get("SLURM_JOB_ID"):
    raise RuntimeError(
        "This is the login-node control-plane notebook. "
        "Do not run it from inside a compute allocation."
    )

core_config.map_storage()
core_config.set_hpc_proxies()
print("Login-node control plane initialized. No heavy computation enabled.")


In [ ]:
print("=== QUOTA CHECK (LIGHTWEIGHT METADATA ONLY) ===")
try:
    result = subprocess.run(
        ["quota_check"],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.stderr:
        print("quota_check stderr:")
        print(result.stderr)
    print(f"exit_code={result.returncode}")
except FileNotFoundError:
    print("quota_check is not installed/available in this environment.")


In [ ]:
print("GPU work is intentionally disabled in this notebook.")
print("Use phase1_test_compute_node.ipynb from an allocated compute node for CUDA/model tests.")


In [ ]:
print("Environment provisioning is intentionally disabled here.")
print("Install packages and build native extensions only inside the allocated compute-node environment.")


In [ ]:
import os

USED_GB = os.environ.get("AIEROS_QUOTA_USED_GB")
LIMIT_GB = os.environ.get("AIEROS_QUOTA_LIMIT_GB")

if USED_GB is None or LIMIT_GB is None:
    print("Set AIEROS_QUOTA_USED_GB and AIEROS_QUOTA_LIMIT_GB from the quota_check result before writing a snapshot.")
    print("Example:")
    print("  export AIEROS_QUOTA_USED_GB=186.3")
    print("  export AIEROS_QUOTA_LIMIT_GB=200")
else:
    print(f"Quota values supplied: used={USED_GB} GB, limit={LIMIT_GB} GB")


In [ ]:
from pathlib import Path
from tools.quota_snapshot import atomic_write
import time

used_gb = float(os.environ["AIEROS_QUOTA_USED_GB"])
limit_gb = float(os.environ["AIEROS_QUOTA_LIMIT_GB"])

if used_gb < 0 or limit_gb <= 0 or used_gb > limit_gb:
    raise ValueError("Invalid quota values: require 0 <= used <= limit and limit > 0")

snapshot_path = Path("~/.cache/aieros/quota_snapshot.json").expanduser()
atomic_write(
    snapshot_path,
    {
        "used_bytes": int(used_gb * 1024**3),
        "limit_bytes": int(limit_gb * 1024**3),
        "timestamp": time.time(),
        "source": "login-node quota_check",
    },
)
print(f"Fresh quota snapshot written atomically to {snapshot_path}")


In [ ]:
print("=== SLURM QUEUE (LIGHTWEIGHT METADATA ONLY) ===")
try:
    result = subprocess.run(
        ["squeue", "-u", os.environ.get("USER", "")],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
except FileNotFoundError:
    print("squeue is not available in this environment.")


In [ ]:
print("No package installation is performed from the login-node notebook.")
print("Use a compute-node setup job for environment provisioning.")


In [ ]:
print("No model inference is performed on the login node.")
print("Use phase1_test_compute_node.ipynb inside an allocated compute node.")


In [ ]:
print("Repository cloning is intentionally disabled on the login node.")
print("Prepare repositories in compute-node jobs or dedicated provisioning workflows.")


In [ ]:
print("Wan/LTX GPU tests are intentionally disabled on the login node.")


In [ ]:
print("Audio model provisioning and inference are intentionally disabled on the login node.")


In [ ]:
print("No AudioGen inference is performed on the login node.")


In [ ]:
print("Hugging Face package installation is intentionally disabled on the login node.")


In [ ]:
print("Cloud upload testing is intentionally disabled here.")
print("Use hf_upload_daemon.py on an allocated compute node; it is quota-aware.")


In [ ]:
print("Run quota_check above after any relevant storage change.")
